In [1]:
!pip install vitaldb numpy pandas torch


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import vitaldb
import pandas as pd
import random
import matplotlib.pyplot as plt

In [3]:
def get_sample(sample_size, track_names=None):
    """
    Pull a random sample of cases from VitalDB using pre-built case sets.
    
    Args:
        sample_size: Number of cases to sample
        track_names: Pre-built set name ('BIS', 'PPF', 'SEVO', 'DES', 'N2O', 'RFT50', 'RFT20', 'TIVA')
                  or list of track names to filter cases by.
                  If None, defaults to 'BIS' cases.
    
    Returns:
        pd.DataFrame: DataFrame with case data for each sampled case
    """
    caseid_sets = {
        'BIS': vitaldb.caseids_bis,
        'PPF': vitaldb.caseids_ppf,
        'SEVO': vitaldb.caseids_sevo,
        'DES': vitaldb.caseids_des,
        'N2O': vitaldb.caseids_n2o,
        'RFT50': vitaldb.caseids_rft50,
        'RFT20': vitaldb.caseids_rft20,
        'TIVA': vitaldb.caseids_tiva
    }
    
    if track_names is None:
        caseids = list(vitaldb.caseids_bis)
    elif isinstance(track_names, str) and track_names.upper() in caseid_sets:
        caseids = list(caseid_sets[track_names.upper()])
    else:
        caseids = vitaldb.find_cases(track_names)
    
    if len(caseids) == 0:
        return pd.DataFrame()
    
    sampled_ids = random.sample(caseids, min(sample_size, len(caseids)))
    
    records = []
    for caseid in sampled_ids:
        vf = vitaldb.VitalFile(caseid)
        case_df = vf.to_pandas(vf.get_track_names(),1)
        case_df['caseid'] = caseid
        records.append(case_df)
    
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


def get_features_sample(features, sample_size):
    """
    Pull a sample of specific features from VitalDB.
    
    Args:
        features: List of track names to extract (e.g., ['ECG_II', 'PLETH', 'BIS', 'ART_MBP'])
        sample_size: Number of cases to sample
    
    Returns:
        pd.DataFrame: DataFrame with specified features for each sampled case
    """
    caseids = vitaldb.find_cases(features)
    if len(caseids) == 0:
        return pd.DataFrame()
    
    sampled_ids = random.sample(caseids, len(caseids))
    
    records = []
    for caseid in sampled_ids:
        vf = vitaldb.VitalFile(caseid)
        try:
            case_df = vf.to_pandas(features)
            case_df['caseid'] = caseid
            records.append(case_df)
        except Exception:
            continue
    
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()

In [4]:
df = get_features_sample(['BIS'], 10)
df

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

def get_bis_data(sample_size=5, track_names='BIS'):
    """Pull BIS data for multiple cases from VitalDB."""
    caseids = vitaldb.find_cases(track_names)
    if len(caseids) == 0:
        print(f"No cases found for {track_names}")
        return pd.DataFrame()
    
    sampled_ids = random.sample(caseids, min(sample_size, len(caseids)))
    print(f"Fetching BIS data for {len(sampled_ids)} cases...")
    
    records = []
    for i, caseid in enumerate(sampled_ids):
        try:
            vf = vitaldb.VitalFile(caseid)
            df = vf.to_pandas(['BIS'], 1)  # 1Hz downsampling
            if 'BIS' in df.columns and not df['BIS'].isna().all():
                df['caseid'] = caseid
                df['time_seconds'] = df.index  # time in seconds
                records.append(df[['time_seconds', 'BIS', 'caseid']])
                print(f"  [{i+1}] Case {caseid}: {len(df)} samples, BIS range: {df['BIS'].min():.1f} - {df['BIS'].max():.1f}")
        except Exception as e:
            print(f"  [{i+1}] Case {caseid}: ERROR - {e}")
            continue
    
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


def plot_bis_trajectories(bis_df, max_cases=5):
    """Plot BIS trajectories - each patient on separate subplot (line by line)."""
    if bis_df.empty:
        print("No data to plot")
        return
    
    caseids = bis_df['caseid'].unique()[:max_cases]
    n_cases = len(caseids)
    
    fig, axes = plt.subplots(n_cases, 1, figsize=(12, 3 * n_cases), sharex=True)
    if n_cases == 1:
        axes = [axes]
    
    for i, caseid in enumerate(caseids):
        ax = axes[i]
        case_data = bis_df[bis_df['caseid'] == caseid].sort_values('time_seconds')
        
        ax.plot(case_data['time_seconds'] / 60, case_data['BIS'], 
               color='blue', linewidth=1)
        
        ax.axhline(y=50, color='red', linestyle='--', linewidth=1.5, label='Target BIS=50')
        ax.axhline(y=40, color='orange', linestyle=':', linewidth=1, alpha=0.5)
        ax.axhline(y=60, color='orange', linestyle=':', linewidth=1, alpha=0.5)
        ax.fill_between([0, case_data['time_seconds'].max() / 60], 40, 60, alpha=0.1, color='green')
        
        ax.set_ylabel('BIS', fontsize=10)
        ax.set_title(f'Case {caseid}', fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 100])
        if i == 0:
            ax.legend(loc='upper right', fontsize=9)
    
    axes[-1].set_xlabel('Time (minutes)', fontsize=12)
    plt.suptitle(f'BIS Trajectories (n={n_cases} patients)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


# Fetch and plot BIS data
print("=" * 60)
print("FETCHING REAL PATIENT BIS DATA FROM VITALDB")
print("=" * 60)

bis_data = get_bis_data(sample_size=5, track_names='BIS')

if not bis_data.empty:
    print(f"\nTotal samples: {len(bis_data)}")
    plot_bis_trajectories(bis_data, max_cases=5)
else:
    print("No BIS data retrieved")